<a href="https://colab.research.google.com/github/Lingeshkumar24-code/NLP-projects/blob/main/NLP_week_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [2]:
data = {
    "Review":[

# Positive Reviews
"The room was clean and spacious. The staff were very friendly and helpful.",
"The breakfast buffet had many delicious options and the service was excellent.",
"I loved the swimming pool and the hotel atmosphere was relaxing.",
"The check-in process was quick and the receptionist was polite.",
"Our family enjoyed the stay because the rooms were comfortable.",
"The housekeeping staff cleaned the room every day.",
"The hotel location was perfect and close to tourist attractions.",
"The bed was very comfortable and I slept peacefully.",
"The restaurant served tasty food with fast service.",
"The hotel offered great value for money with excellent facilities.",

# Negative Reviews
"The room was dirty and the bathroom smelled bad.",
"The air conditioner was not working during the entire stay.",
"The hotel staff behaved rudely and ignored our complaints.",
"The food quality was poor and dinner was served cold.",
"The room service was extremely slow.",
"The bedsheets were stained and not changed.",
"There was too much noise during the night.",
"The WiFi connection was very slow.",
"The swimming pool was closed without notice.",
"The parking area was overcrowded and unsafe.",

# More Positive
"The staff upgraded our room without any extra charge.",
"The hotel provided free airport pickup which was very convenient.",
"The room had an amazing city view.",
"The gym equipment was modern and clean.",
"The restaurant staff were friendly and attentive.",
"The hotel was beautifully decorated.",
"The bathroom was clean with fresh towels.",
"The complimentary breakfast exceeded expectations.",
"The children enjoyed the play area.",
"The hotel management responded quickly to requests.",

# More Negative
"The receptionist took a long time to complete check-in.",
"The bathroom had leaking pipes.",
"The room was too small for the price.",
"The elevator stopped working several times.",
"The staff were unprofessional during checkout.",
"The food was overpriced and tasteless.",
"The hotel was not properly maintained.",
"The air conditioning produced loud noise.",
"The towels were old and dirty.",
"The internet disconnected frequently."

],

"Sentiment":[

"Positive","Positive","Positive","Positive","Positive",
"Positive","Positive","Positive","Positive","Positive",

"Negative","Negative","Negative","Negative","Negative",
"Negative","Negative","Negative","Negative","Negative",

"Positive","Positive","Positive","Positive","Positive",
"Positive","Positive","Positive","Positive","Positive",

"Negative","Negative","Negative","Negative","Negative",
"Negative","Negative","Negative","Negative","Negative"

]
}

df = pd.DataFrame(data)

df.head()

,Review,Sentiment
0,The room was clean and spacious. The staff wer...,Positive
1,The breakfast buffet had many delicious option...,Positive
2,I loved the swimming pool and the hotel atmosp...,Positive
3,The check-in process was quick and the recepti...,Positive
4,Our family enjoyed the stay because the rooms ...,Positive


In [3]:

print(df.shape)

print(df["Sentiment"].value_counts())

(40, 2)
Sentiment
Positive    20
Negative    20
Name: count, dtype: int64


In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess(text):

    text = text.lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    words = text.split()

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

df["Clean_Review"] = df["Review"].apply(preprocess)

df.head()

,Review,Sentiment,Clean_Review
0,The room was clean and spacious. The staff wer...,Positive,room clean spacious staff friendly helpful
1,The breakfast buffet had many delicious option...,Positive,breakfast buffet many delicious option service...
2,I loved the swimming pool and the hotel atmosp...,Positive,loved swimming pool hotel atmosphere relaxing
3,The check-in process was quick and the recepti...,Positive,checkin process quick receptionist polite
4,Our family enjoyed the stay because the rooms ...,Positive,family enjoyed stay room comfortable


In [5]:
X = df["Clean_Review"]

y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
model = Pipeline([

    ("tfidf", TfidfVectorizer()),

    ("classifier", MultinomialNB())

])

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [7]:
predictions = model.predict(X_test)

predictions

array(['Positive', 'Positive', 'Positive', 'Negative', 'Positive',
       'Positive', 'Negative', 'Positive'], dtype='<U8')

In [8]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.375


In [9]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

    Negative       0.50      0.20      0.29         5
    Positive       0.33      0.67      0.44         3

    accuracy                           0.38         8
   macro avg       0.42      0.43      0.37         8
weighted avg       0.44      0.38      0.35         8



In [10]:
issues = {
    "Room": ["room", "bed", "bathroom", "towel", "bedsheet"],
    "Staff": ["staff", "receptionist", "housekeeping", "service"],
    "Food": ["food", "restaurant", "breakfast", "dinner"],
    "Amenities": ["wifi", "pool", "gym", "parking", "air", "elevator"]
}

issue_count = {
    "Room":0,
    "Staff":0,
    "Food":0,
    "Amenities":0
}

negative_reviews = df[df["Sentiment"]=="Negative"]["Clean_Review"]

for review in negative_reviews:
    for category, keywords in issues.items():
        if any(word in review for word in keywords):
            issue_count[category] += 1

issue_df = pd.DataFrame(issue_count.items(), columns=["Service Area","Number of Complaints"])

issue_df

,Service Area,Number of Complaints
0,Room,6
1,Staff,4
2,Food,2
3,Amenities,6


In [13]:
positive = len(df[df["Sentiment"]=="Positive"])
negative = len(df[df["Sentiment"]=="Negative"])
total = len(df)

satisfaction = (positive / total) * 100

print("Total Reviews :", total)
print("Positive Reviews :", positive)
print("Negative Reviews :", negative)
print("Customer Satisfaction : {:.2f}%".format(satisfaction))

print("\nMost Common Service Issues")
print(issue_df)

Total Reviews : 40
Positive Reviews : 20
Negative Reviews : 20
Customer Satisfaction : 50.00%

Most Common Service Issues
  Service Area  Number of Complaints
0         Room                     6
1        Staff                     4
2         Food                     2
3    Amenities                     6


In [14]:
new_reviews = [

"The room was very clean and the staff were extremely helpful.",

"The food was terrible and the waiter was rude.",

"The swimming pool and gym were excellent.",

"The bathroom was dirty and WiFi was not working.",

"The hotel location was fantastic and breakfast was delicious."

]

result = model.predict(new_reviews)

for review, sentiment in zip(new_reviews, result):
    print("Review :", review)
    print("Predicted Sentiment :", sentiment)

Review : The room was very clean and the staff were extremely helpful.
Predicted Sentiment : Positive
Review : The food was terrible and the waiter was rude.
Predicted Sentiment : Negative
Review : The swimming pool and gym were excellent.
Predicted Sentiment : Positive
Review : The bathroom was dirty and WiFi was not working.
Predicted Sentiment : Negative
Review : The hotel location was fantastic and breakfast was delicious.
Predicted Sentiment : Positive
